# 🧫 Raw Milk Bacterial Contamination & Antibiotic Resistance Analysis

**Study Objectives:**
1. Assess bacterial communities isolated from raw milk
2. Determine antibiotic susceptibility patterns against clinically relevant antibiotics
3. Analyse risk factors contributing to bacterial contamination in raw milk

**Datasets used:**
- `Isolated_bacteria_Milk_Study_Data.xlsx` — Microbiology isolation data (n=25)
- `raw_milk_data_questionaire_response.xlsx` — Farm hygiene/practice questionnaire (n=25)
- `AST_results_1_.xlsx` — Antibiotic susceptibility testing (gram-neg n=21; gram-pos n=4)

> **How to use:** Upload all 3 Excel files when Step 1 prompts you, then `Runtime → Run all`.

## STEP 0 — Install & Import Libraries

In [ ]:
# Install libraries not bundled in Colab
!pip install -q pingouin openpyxl statsmodels

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
import statsmodels.api as sm
from statsmodels.formula.api import logit
import pingouin as pg
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings, itertools
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 5)})
print('All libraries loaded successfully.')

## STEP 1 — Upload & Load Data

In [ ]:
from google.colab import files
print('Upload all 3 Excel files:')
print('  1. Isolated_bacteria_Milk_Study_Data.xlsx')
print('  2. raw_milk_data_questionaire_response.xlsx')
print('  3. AST_results_1_.xlsx')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Microbiology data ───────────────────────────────────────────────────────
df_micro = pd.read_excel('Isolated_bacteria_Milk_Study_Data.xlsx',
                         sheet_name='Microbiology_Data')
df_micro.columns = df_micro.columns.str.strip()
df_micro['Source'] = df_micro['Source'].str.strip()
df_micro['Sample_ID'] = df_micro['Sample_ID'].str.strip()
print(f'Microbiology data: {df_micro.shape}')
df_micro.head()

In [ ]:
# ── Questionnaire data ──────────────────────────────────────────────────────
df_q = pd.read_excel('raw_milk_data_questionaire_response.xlsx',
                     sheet_name='questionaire_response')
df_q.columns = df_q.columns.str.strip()
col_map = {
    'No.': 'No', 'Sample ID': 'Sample_ID', 'Date (DD/MM/YYYY)': 'Date',
    'Q1 barn_ cleaning': 'barn_cleaning', 'Q2 udder_ wash': 'udder_wash',
    'Q3 utensil_ freq': 'utensil_freq', 'Q4 utensil_ method': 'utensil_method',
    'Q5 trad_ flavor': 'trad_flavor', 'Q6 handwash_ before': 'handwash_before',
    'Q7 handwash_ soap': 'handwash_soap', 'Q8 udder_ wash_freq': 'udder_wash_freq',
    'Q9a plain_ water': 'q9_plain', 'Q9b warm_ water': 'q9_warm',
    'Q9c soap_ deterg': 'q9_soap', 'Q9d sanitizer': 'q9_sanitizer',
    'Q9e no_ washing': 'q9_none', 'Q10 teat_ drying': 'teat_drying',
    'Q11 container_ type': 'container_type',
    'Q12 equip_ clean_freq': 'equip_clean_freq',
    'Q13 equip_ clean_meth': 'equip_clean_method'
}
df_q = df_q.rename(columns=col_map).dropna(subset=['No'])
df_q['Sample_ID'] = df_q['Sample_ID'].astype(str).str.strip()
print(f'Questionnaire data: {df_q.shape}')
df_q.head()

In [ ]:
# ── AST results ─────────────────────────────────────────────────────────────
df_gn = pd.read_excel('AST_results_1_.xlsx', sheet_name='gram_negative_ast_results')
df_gp = pd.read_excel('AST_results_1_.xlsx', sheet_name='gram_positive_AST_results')
df_gn.columns = df_gn.columns.str.strip().str.replace(r'\s*\(\d+\)', '', regex=True)
df_gp.columns = df_gp.columns.str.strip()
df_gn['Gram'] = 'Negative'
df_gp['Gram'] = 'Positive'

ab_cols_gn = ['CPR', 'MER', 'AMK', 'CTR', 'AUG']
ab_cols_gp = ['CIP', 'Gen', 'Pen', 'Eryth']

# Clean result values (strip asterisks, spaces)
for c in ab_cols_gn:
    df_gn[c] = df_gn[c].astype(str).str.strip().str.replace(r'[^SIRNTsirnt]', '', regex=True).str.upper()
for c in ab_cols_gp:
    df_gp[c] = df_gp[c].astype(str).str.strip().str.replace(r'[^SIRNTsirnt]', '', regex=True).str.upper()

print(f'Gram-negative isolates: {len(df_gn)}')
print(f'Gram-positive isolates : {len(df_gp)}')
display(df_gn.head())
display(df_gp.head())

In [ ]:
# ── Merge microbiology + questionnaire ──────────────────────────────────────
bacteria_cols = ['E_coli', 'P_aeruginosa', 'S_aureus', 'Klebsiella',
                 'Salmonella', 'Proteus_sp', 'Enterobacteriaceae_family']

df_micro_s = df_micro.reset_index(drop=True)
df_q_s     = df_q.reset_index(drop=True)

df_merged = df_micro_s.merge(
    df_q_s.drop(columns=['No', 'Date'], errors='ignore'),
    left_index=True, right_index=True, suffixes=('_micro', '_q')
)
df_merged['contaminated'] = df_merged[bacteria_cols].max(axis=1)

print(f'Merged dataset: {df_merged.shape}')
print(f'Contaminated : {df_merged["contaminated"].sum()}')
print(f'No growth    : {(df_merged["contaminated"]==0).sum()}')
df_merged[['Sample_ID', 'Source', 'Isolated_Organisms', 'contaminated']].head(10)

---
## OBJECTIVE I — Bacterial Communities in Raw Milk
### 1.1 Prevalence Table

In [ ]:
org_labels = {
    'E_coli': 'E. coli',
    'P_aeruginosa': 'P. aeruginosa',
    'S_aureus': 'S. aureus',
    'Klebsiella': 'Klebsiella spp.',
    'Salmonella': 'Salmonella spp.',
    'Proteus_sp': 'Proteus spp.',
    'Enterobacteriaceae_family': 'Enterobacteriaceae family'
}
total_n = len(df_micro)

prev_rows = []
for col, label in org_labels.items():
    count = int(df_micro[col].sum())
    prev_rows.append({'Organism': label, 'Count (n)': count,
                      'Prevalence (%)': round(count/total_n*100, 1)})

df_prev = pd.DataFrame(prev_rows).sort_values('Count (n)', ascending=False)
n_contam = int((df_micro[bacteria_cols].max(axis=1)==1).sum())
df_prev.loc[len(df_prev)] = ['ANY isolate (contaminated)', n_contam,
                              round(n_contam/total_n*100, 1)]
print(f'Total samples: {total_n}')
df_prev

In [ ]:
# Prevalence by source
source_prev = []
for src in ['Farm', 'Market']:
    sub = df_micro[df_micro['Source'] == src]
    n   = len(sub)
    for col, label in org_labels.items():
        cnt = int(sub[col].sum())
        source_prev.append({'Source': src, 'Organism': label, 'n_source': n,
                            'Count': cnt, 'Prevalence (%)': round(cnt/n*100, 1)})

df_src_prev = pd.DataFrame(source_prev)
pivot_src = df_src_prev.pivot_table(index='Organism', columns='Source',
                                     values='Prevalence (%)', aggfunc='first')
print('Prevalence (%) by source:')
pivot_src

### 1.2 Visualisations

In [ ]:
# Fig 1: Grouped bar — prevalence by source
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
df_prev_plot = df_prev[df_prev['Organism'] != 'ANY isolate (contaminated)'].copy()

bars = axes[0].barh(df_prev_plot['Organism'], df_prev_plot['Prevalence (%)'],
                    color=sns.color_palette('muted', len(df_prev_plot)))
axes[0].set_xlabel('Prevalence (%)')
axes[0].set_title('Overall Bacterial Prevalence (n=25)', fontweight='bold')
for bar, val in zip(bars, df_prev_plot['Prevalence (%)']):
    axes[0].text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val}%', va='center', fontsize=9)

df_src_wide = df_src_prev.pivot_table(index='Organism', columns='Source',
                                       values='Prevalence (%)').reset_index()
x = np.arange(len(df_src_wide))
w = 0.35
axes[1].bar(x-w/2, df_src_wide.get('Farm',  pd.Series([0]*len(df_src_wide))), w,
            label='Farm',   color='#4C72B0')
axes[1].bar(x+w/2, df_src_wide.get('Market', pd.Series([0]*len(df_src_wide))), w,
            label='Market', color='#DD8452')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_src_wide['Organism'], rotation=35, ha='right', fontsize=9)
axes[1].set_ylabel('Prevalence (%)')
axes[1].set_title('Prevalence by Source', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('fig1_prevalence.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 2: Interactive donut pie
df_pie = df_prev[df_prev['Count (n)'] > 0]
df_pie = df_pie[df_pie['Organism'] != 'ANY isolate (contaminated)']
fig_pie = px.pie(df_pie, values='Count (n)', names='Organism',
                 title='Distribution of Isolated Bacterial Species',
                 hole=0.35, color_discrete_sequence=px.colors.qualitative.Set2)
fig_pie.update_traces(textinfo='label+percent', pull=[0.05]*len(df_pie))
fig_pie.show()

In [ ]:
# Fig 3: Presence/absence heatmap
heat_data = df_micro.set_index('Sample_ID')[bacteria_cols].copy()
heat_data.columns = [org_labels[c] for c in bacteria_cols]
plt.figure(figsize=(12, 9))
sns.heatmap(heat_data.astype(int), cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': '1 = Isolated'},
            yticklabels=df_micro['Sample_ID'] + ' [' + df_micro['Source'] + ']')
plt.title('Presence/Absence per Sample', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('fig3_heatmap.png', bbox_inches='tight')
plt.show()

---
## OBJECTIVE II — Antibiotic Susceptibility Patterns
### 2.1 S/I/R Distributions

In [ ]:
def ast_summary(df, ab_cols, gram_label):
    rows = []
    for ab in ab_cols:
        counts = df[ab].value_counts()
        nt = counts.get('NT', 0)
        tested = len(df) - nt
        for cat in ['S', 'I', 'R', 'NT']:
            n = counts.get(cat, 0)
            denom = tested if cat != 'NT' else len(df)
            rows.append({'Gram': gram_label, 'Antibiotic': ab, 'Category': cat,
                         'n': int(n), 'Tested_excl_NT': tested,
                         'Pct': round(n/denom*100, 1) if denom > 0 else 0})
    return pd.DataFrame(rows)

df_gn_sum = ast_summary(df_gn, ab_cols_gn, 'Gram-negative')
df_gp_sum = ast_summary(df_gp, ab_cols_gp, 'Gram-positive')

print('=== GRAM-NEGATIVE ===')
print('CPR=Ciprofloxacin, MER=Meropenem, AMK=Amikacin, CTR=Ceftriaxone, AUG=Amoxicillin-Clavulanate')
display(df_gn_sum.pivot_table(index='Antibiotic', columns='Category',
                               values='Pct', aggfunc='first').fillna(0)[['S','I','R','NT']])

print('\n=== GRAM-POSITIVE ===')
print('CIP=Ciprofloxacin, Gen=Gentamicin, Pen=Penicillin, Eryth=Erythromycin')
display(df_gp_sum.pivot_table(index='Antibiotic', columns='Category',
                               values='Pct', aggfunc='first').fillna(0)[['S','I','R','NT']])

In [ ]:
# MDR classification (resistant to >= 3 antibiotic classes)
def count_R(row, cols):
    return sum(1 for c in cols if row.get(c, 'NT') == 'R')

df_gn['n_resistant'] = df_gn.apply(lambda r: count_R(r, ab_cols_gn), axis=1)
df_gn['MDR'] = df_gn['n_resistant'] >= 3
df_gp['n_resistant'] = df_gp.apply(lambda r: count_R(r, ab_cols_gp), axis=1)
df_gp['MDR'] = df_gp['n_resistant'] >= 3

print('GRAM-NEGATIVE MDR summary:')
display(df_gn[['Sample ID', 'Organism', 'n_resistant', 'MDR']])
print(f'MDR isolates: {df_gn["MDR"].sum()}/{len(df_gn)} ({df_gn["MDR"].mean()*100:.1f}%)')

print('\nGRAM-POSITIVE MDR summary:')
display(df_gp[['Sample ID', 'Organism', 'n_resistant', 'MDR']])

In [ ]:
# Resistance rate per organism
org_res = []
for org in df_gn['Organism'].unique():
    sub = df_gn[df_gn['Organism'] == org]
    for ab in ab_cols_gn:
        tested = sub[sub[ab] != 'NT']
        if len(tested) == 0:
            continue
        rate = (tested[ab] == 'R').sum() / len(tested) * 100
        org_res.append({'Organism': org, 'Antibiotic': ab,
                        'n_tested': len(tested), 'Resistance_%': round(rate, 1)})

df_org_res = pd.DataFrame(org_res)
pivot_org = df_org_res.pivot_table(index='Organism', columns='Antibiotic',
                                    values='Resistance_%').fillna(np.nan)
print('Resistance (%) by organism — Gram-negative:')
pivot_org

### 2.2 Visualisations

In [ ]:
# Fig 4: Stacked bar S/I/R
cat_colors = {'S': '#2ecc71', 'I': '#f39c12', 'R': '#e74c3c', 'NT': '#bdc3c7'}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (df_sum, title) in zip(axes, [
        (df_gn_sum, 'Gram-Negative\n(CPR, MER, AMK, CTR, AUG)'),
        (df_gp_sum, 'Gram-Positive\n(CIP, Gen, Pen, Eryth)')]):
    pivot = df_sum.pivot_table(index='Antibiotic', columns='Category',
                               values='Pct', aggfunc='first').fillna(0)
    pivot = pivot.reindex(columns=['S','I','R','NT'], fill_value=0)
    bottom = np.zeros(len(pivot))
    for cat in ['S','I','R','NT']:
        vals = pivot[cat].values
        ax.bar(pivot.index, vals, bottom=bottom,
               color=cat_colors[cat], label=cat, edgecolor='white')
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                ax.text(i, b+v/2, f'{v:.0f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottom += vals
    ax.set_ylim(0, 100)
    ax.set_ylabel('Proportion (%)')
    ax.set_title(title, fontweight='bold')
    ax.legend(title='Result', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('fig4_AST_stacked.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 5: Resistance heatmap by organism
plt.figure(figsize=(10, 6))
mask = pivot_org.isna()
sns.heatmap(pivot_org, annot=True, fmt='.0f', cmap='Reds',
            linewidths=0.5, linecolor='white', mask=mask,
            cbar_kws={'label': 'Resistance (%)'})
plt.title('Resistance (%) by Organism — Gram-Negative', fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_resistance_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 6: Interactive radar chart — resistance profiles
org_radar = df_org_res.pivot_table(index='Organism', columns='Antibiotic',
                                    values='Resistance_%', aggfunc='first').fillna(0).reset_index()
categories = ab_cols_gn
fig_radar = go.Figure()
for _, row in org_radar.iterrows():
    vals = [row.get(c, 0) for c in categories]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=categories + [categories[0]],
        fill='toself', name=row['Organism'], opacity=0.6
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Resistance Profiles by Organism (Gram-Negative)'
)
fig_radar.show()

---
## OBJECTIVE III — Risk Factor Analysis
### 3.1 Descriptive — Hygiene Practice Frequencies

In [ ]:
label_maps = {
    'barn_cleaning'    : {1:'2x/day', 2:'1x/day', 3:'3x/week', 4:'1x/week'},
    'udder_wash'       : {1:'No wash', 2:'Warm tap', 3:'Cold tap',
                          4:'Warm river', 5:'Cold river'},
    'utensil_freq'     : {1:'2x/day', 2:'Daily', 3:'Every other day'},
    'utensil_method'   : {1:'Hot water', 2:'Cold water'},
    'handwash_before'  : {0:'No', 1:'Yes'},
    'handwash_soap'    : {1:'Always', 2:'Sometimes', 3:'Never'},
    'udder_wash_freq'  : {1:'Always', 2:'Sometimes', 3:'Never'},
    'teat_drying'      : {1:'Always', 2:'Sometimes', 3:'Never'},
    'container_type'   : {1:'Stainless steel', 2:'Food-grade plastic',
                          3:'Non-food-grade plastic', 4:'Other'},
    'equip_clean_freq' : {1:'After every milking', 2:'Once/day',
                          3:'Once/week', 4:'Rarely/Never'},
    'equip_clean_method': {1:'Hot water+detergent', 2:'Cold water only',
                           3:'Soap w/o hot water', 4:'No cleaning'}
}
practice_cols = list(label_maps.keys())

for col in practice_cols:
    freq = df_merged[col].value_counts().sort_index()
    pct  = (freq / len(df_merged) * 100).round(1)
    tbl  = pd.DataFrame({'n': freq, '%': pct})
    tbl.index = tbl.index.map(label_maps[col])
    print(f'\n── {col.upper()} ──')
    print(tbl.to_string())

### 3.2 Chi-Square / Fisher's Exact Tests

In [ ]:
def chi_or_fisher(var, df, outcome='contaminated'):
    ct = pd.crosstab(df[var], df[outcome])
    expected = chi2_contingency(ct)[3]
    if (expected < 5).any():
        if ct.shape == (2, 2):
            OR, p = fisher_exact(ct)
            return p, "Fisher's exact", ct, OR
        else:
            chi2, p, dof, _ = chi2_contingency(ct, correction=True)
            return p, 'Chi-sq (Yates, low exp counts)', ct, None
    else:
        chi2, p, dof, _ = chi2_contingency(ct)
        return p, 'Chi-square', ct, None

results = []
for col in practice_cols:
    p, test, ct, OR = chi_or_fisher(col, df_merged)
    results.append({'Variable': col, 'Test': test,
                    'p-value': round(p, 4),
                    'Sig (p<0.05)': 'Yes' if p < 0.05 else 'No',
                    'OR (if 2x2)': round(OR, 3) if OR else 'N/A'})

df_chi = pd.DataFrame(results).sort_values('p-value')
print('Associations between farm practices and milk contamination:')
df_chi

In [ ]:
# Detailed cross-tabs for significant variables
sig_vars = df_chi[df_chi['Sig (p<0.05)'] == 'Yes']['Variable'].tolist()
if sig_vars:
    for var in sig_vars:
        print(f'\n=== {var} ===')
        ct = pd.crosstab(df_merged[var].map(label_maps[var]),
                         df_merged['contaminated'],
                         margins=True, margins_name='Total')
        ct.columns = ['Not contaminated', 'Contaminated', 'Total']
        print(ct)
else:
    print('No significant associations at p<0.05 (likely due to small n=25).')
    print('All results are exploratory. Report p-values and note the limitation.')

In [ ]:
# Farm vs Market association
ct_src = pd.crosstab(df_merged['Source'], df_merged['contaminated'])
ct_src.columns = ['Not contaminated', 'Contaminated']
print('Farm vs Market cross-tab:')
print(ct_src)
if ct_src.shape == (2, 2):
    OR, p = fisher_exact(ct_src)
    print(f"\nFisher's Exact: OR={OR:.3f}, p={p:.4f}")
else:
    chi2, p, dof, _ = chi2_contingency(ct_src)
    print(f'Chi-square: chi2={chi2:.3f}, p={p:.4f}')

### 3.3 Logistic Regression

In [ ]:
# Univariable logistic regression for each practice variable
lr_results = []
for col in practice_cols:
    try:
        model = logit(f'contaminated ~ C({col})', data=df_merged).fit(
            disp=0, method='bfgs', maxiter=200)
        for param in model.params.index:
            if param == 'Intercept':
                continue
            OR  = np.exp(model.params[param])
            ci  = np.exp(model.conf_int().loc[param])
            p   = model.pvalues[param]
            lr_results.append({'Variable': col, 'Level': param,
                               'OR': round(OR, 3),
                               'CI_lower': round(ci[0], 3),
                               'CI_upper': round(ci[1], 3),
                               'p': round(p, 4)})
    except Exception as e:
        lr_results.append({'Variable': col, 'Level': 'error: ' + str(e),
                           'OR': np.nan, 'CI_lower': np.nan,
                           'CI_upper': np.nan, 'p': np.nan})

df_lr = pd.DataFrame(lr_results).sort_values('p')
print('Univariable Logistic Regression (outcome = contaminated):')
df_lr

In [ ]:
# Multivariable logistic regression (exploratory — n=25 is small)
df_merged['source_binary'] = (df_merged['Source'] == 'Farm').astype(int)
multi_vars = ['handwash_before', 'udder_wash_freq', 'teat_drying', 'container_type']
formula = 'contaminated ~ source_binary + ' + ' + '.join([f'C({v})' for v in multi_vars])

try:
    model_m = logit(formula, data=df_merged).fit(disp=0, method='bfgs', maxiter=300)
    or_tbl = pd.DataFrame({
        'OR': np.exp(model_m.params).round(3),
        'CI_lower': np.exp(model_m.conf_int()[0]).round(3),
        'CI_upper': np.exp(model_m.conf_int()[1]).round(3),
        'p': model_m.pvalues.round(4)
    })
    print('Multivariable Odds Ratio Table:')
    display(or_tbl)
    print(model_m.summary2())
except Exception as e:
    print(f'Model did not converge: {e}')
    print('Reduce predictors or report univariable results only (appropriate for n=25).')

### 3.4 Visualisations — Risk Factors

In [ ]:
# Fig 7: Forest plot
df_forest = df_lr.dropna(subset=['OR','p']).copy()
df_forest = df_forest[~df_forest['Level'].str.startswith('error')]
df_forest['label'] = (df_forest['Variable'] + ' | ' +
    df_forest['Level'].str.replace(r'C\(.*?\)\[T\.', '', regex=True).str.rstrip(']'))
df_forest = df_forest.sort_values('OR')

fig, ax = plt.subplots(figsize=(10, max(6, len(df_forest)*0.45)))
y = range(len(df_forest))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in df_forest['p']]
ax.hlines(y=list(y), xmin=df_forest['CI_lower'].values,
          xmax=df_forest['CI_upper'].values, colors=colors, linewidth=2)
ax.scatter(df_forest['OR'], list(y), color=colors, s=60, zorder=5)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax.set_yticks(list(y))
ax.set_yticklabels(df_forest['label'], fontsize=8)
ax.set_xlabel('Odds Ratio (95% CI)')
ax.set_title('Forest Plot — Univariable Logistic Regression\n(red = p<0.05)', fontweight='bold')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('fig7_forest_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 8: Contamination rate by key hygiene practices
key_vars = ['handwash_before', 'udder_wash_freq', 'teat_drying', 'container_type']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, var in zip(axes, key_vars):
    rate   = df_merged.groupby(var)['contaminated'].mean() * 100
    labels = [label_maps[var].get(k, str(k)) for k in rate.index]
    counts = df_merged[var].value_counts().reindex(rate.index)
    bars   = ax.bar(labels, rate.values,
                    color=sns.color_palette('coolwarm_r', len(rate)))
    ax.set_ylim(0, 115)
    ax.set_ylabel('Contamination rate (%)')
    ax.set_title(var.replace('_', ' ').title(), fontweight='bold')
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
    for bar, n in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+2, f'n={n}', ha='center', fontsize=8)

plt.suptitle('Contamination Rate by Hygiene Practice', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig8_risk_factors.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 9: Significance chart (interactive)
df_chi_p = df_chi.copy()
df_chi_p['-log10p'] = -np.log10(df_chi_p['p-value'].clip(lower=1e-6))
fig9 = px.bar(df_chi_p.sort_values('-log10p'),
              x='-log10p', y='Variable', orientation='h',
              color='Sig (p<0.05)',
              color_discrete_map={'Yes': '#e74c3c', 'No': '#95a5a6'},
              title='Risk Factor Significance (-log10 p | dashed line = p=0.05)')
fig9.add_vline(x=-np.log10(0.05), line_dash='dash', line_color='black')
fig9.show()

---
## STEP 4 — Export All Results

In [ ]:
output_file = 'Raw_Milk_Analysis_Results.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_prev.to_excel(writer, sheet_name='Bacterial_Prevalence', index=False)
    df_src_prev.to_excel(writer, sheet_name='Prevalence_by_Source', index=False)
    df_gn_sum.to_excel(writer, sheet_name='AST_GramNeg', index=False)
    df_gp_sum.to_excel(writer, sheet_name='AST_GramPos', index=False)
    pivot_org.reset_index().to_excel(writer, sheet_name='Resistance_by_Organism', index=False)
    df_gn[['Sample ID','Organism','n_resistant','MDR']].to_excel(
        writer, sheet_name='MDR_Classification', index=False)
    df_chi.to_excel(writer, sheet_name='ChiSq_RiskFactors', index=False)
    df_lr.to_excel(writer, sheet_name='Logistic_Regression', index=False)
    df_merged.to_excel(writer, sheet_name='Merged_Dataset', index=False)

print(f'Exported: {output_file}')
files.download(output_file)

for f in ['fig1_prevalence.png','fig3_heatmap.png','fig4_AST_stacked.png',
           'fig5_resistance_heatmap.png','fig7_forest_plot.png',
           'fig8_risk_factors.png']:
    files.download(f)
    print(f'Downloaded: {f}')

---
## Interpretation Guide

### Objective I
Report count and % prevalence per organism overall and by Farm vs Market. Note which species dominate and whether source affects prevalence.

### Objective II
- Report %S/%I/%R **excluding NT isolates from the denominator**.
- Flag MDR isolates (resistant to ≥3 classes).
- **Universal MER (Meropenem) resistance in all gram-negatives is a critical public-health finding — highlight this prominently.**

### Objective III
- Chi-square / Fisher: p<0.05 = statistically significant association.
- Logistic regression: OR>1 = higher contamination odds; OR<1 = protective.
- ⚠ With n=25 the study is **underpowered** for multivariable regression. Label all findings **exploratory/preliminary** and recommend a larger follow-up study.

### Multiple Comparisons
Consider **Bonferroni correction**: divide α=0.05 by number of tests (e.g., 11 variables → p<0.0045 for significance).